# Taming the Loss Landscape of PINNs with Noisy Feynman-Kac Supervision

**Paper:** Tepakbong, N., Hu, H., Liu, C., Zhou, X. (2026). *Taming the Loss Landscape of PINNs with Noisy Feynman-Kac Supervision: Operator Preconditioning and Non-Asymptotic Error Bounds.* Proceedings of ICML 2026 (PMLR 306).

**Carpeta origen:** `PINNs/4. Otros/Taming the Loss Landscape of PINNs with Noisy Feynman–Kac Supervision.pdf`

## Como se usan las PINNs en este paper

El paper diagnostica que el mal entrenamiento de las PINNs en problemas dificiles se debe a un **mal condicionamiento del operador** subyacente (la curvatura de la perdida hereda el espectro de $\mathcal{L}^*\mathcal{L}$), y propone **FK-PINNs**: aumentar la perdida estandar de PINN (Eq. 2.3-2.5) con un **termino de supervision escaso basado en la representacion de Feynman-Kac (FK)** de la solucion (Eq. 3.1):

$$u^\star(x)=\mathbb{E}_x\Big[\int_0^\tau r(X_t)\,dt+h(X_\tau)\Big]$$

donde $(X_t)$ es un proceso de difusion con generador $\mathcal{L}$ y $\tau$ el tiempo de salida del dominio. Esta expectativa se aproxima por **Monte Carlo** simulando trayectorias Euler-Maruyama (Algoritmo 1) en un pequeno conjunto de puntos $\{x_k^{FK}\}$, dando etiquetas *ruidosas* $\hat u^{MC}(x_k^{FK})$ que se usan como anclas adicionales en la perdida (Eq. 4.1-4.2):

$$\mathcal{R}_{FK-PINN}=\lambda_{PDE}\mathcal{R}_{PDE}+\lambda_{\partial\Omega}\mathcal{R}_{\partial\Omega}+\lambda_{FK}\mathcal{R}_{FK},\qquad \mathcal{R}_{FK}=\frac{1}{N_{FK}}\sum_k\big(u_\theta(x_k^{FK})-\hat u^{MC}(x_k^{FK})\big)^2$$

El paper demuestra (Teorema 5.4) que este termino de supervision actua como un **precondicionador del operador**: el numero de condicion de la perdida PINN estandar empeora polinomicamente con el numero de puntos de colocacion ($\kappa_{PINN}\geq cN^{\beta/2}$), mientras que el de FK-PINN permanece **acotado**, independientemente de $N$ &mdash; una propiedad puramente arquitectonica de la perdida, agnostica al origen exacto de los datos de supervision.

Este cuaderno reproduce fielmente el mecanismo completo para la **ecuacion de Poisson 2D** (uno de los benchmarks del paper): $-\Delta u=f$ en $[0,1]^2$, $u=0$ en la frontera, con solucion exacta $u^\star(x,y)=\sin(\pi x)\sin(\pi y)$. La representacion FK correspondiente usa movimiento Browniano $X_t=x+\sqrt2\,W_t$ (generador $\Delta$), simulado con el esquema Euler-Maruyama del Algoritmo 1 para generar las etiquetas de supervision dispersas y ruidosas, comparando **PINN estandar vs. FK-PINN**.

## Repositorio publico de referencia

El PDF no incluye un repositorio de codigo propio en las paginas revisadas, ni se encontro uno especifico. Como referencia general del framework PINN base:

- **maziarraissi/PINNs** &mdash; https://github.com/maziarraissi/PINNs

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Problema de Poisson 2D: $-\Delta u=f$, $u=0$ en $\partial[0,1]^2$, $u^\star=\sin(\pi x)\sin(\pi y)$

In [ ]:
def exact_u(x, y):
    return np.sin(np.pi * x) * np.sin(np.pi * y)

def f_source(x, y):
    return 2 * np.pi**2 * np.sin(np.pi * x) * np.sin(np.pi * y)

## 2. Algoritmo 1: estimador Monte Carlo de Feynman-Kac (movimiento Browniano, generador $\Delta$)

In [ ]:
def fk_monte_carlo(x0, y0, n_mc=200, dt=0.0025, n_steps_max=800):
    """Algoritmo 1: X_t = (x0,y0) + sqrt(2) W_t (generador = Laplaciano), r=f, h=g=0."""
    X = np.full(n_mc, x0); Y = np.full(n_mc, y0)
    S = np.zeros(n_mc)
    active = np.ones(n_mc, dtype=bool)
    for _ in range(n_steps_max):
        if not active.any():
            break
        dW_x = np.random.randn(n_mc) * np.sqrt(dt)
        dW_y = np.random.randn(n_mc) * np.sqrt(dt)
        X_new = X + np.sqrt(2) * dW_x
        Y_new = Y + np.sqrt(2) * dW_y
        S[active] += f_source(X[active], Y[active]) * dt   # acumulacion de r(X_t) dt
        X, Y = np.where(active, X_new, X), np.where(active, Y_new, Y)
        active = active & (X > 0) & (X < 1) & (Y > 0) & (Y < 1)  # sale del dominio -> deja de acumular
    return S.mean()  # h=g=0 en la salida, por lo que no se anade termino terminal


N_FK = 12  # numero de puntos de supervision FK (escaso, como en el paper)
xy_fk_np = np.random.uniform(0.15, 0.85, (N_FK, 2))  # lejos de la frontera para exit-time razonable
u_fk_labels = np.array([fk_monte_carlo(x, y) for x, y in xy_fk_np])
u_fk_exact = exact_u(xy_fk_np[:, 0], xy_fk_np[:, 1])
print('Etiquetas FK (Monte Carlo, ruidosas) vs. valor exacto en esos puntos:')
for i in range(N_FK):
    print(f'  x={xy_fk_np[i,0]:.2f}, y={xy_fk_np[i,1]:.2f} | FK-MC={u_fk_labels[i]:.4f} | exacto={u_fk_exact[i]:.4f}')

## 3. Redes y perdidas: PINN estandar (Eq. 2.5) vs. FK-PINN (Eq. 4.2)

In [ ]:
class PoissonPINN(nn.Module):
    def __init__(self, n_hidden=4, n_neurons=50):
        super().__init__()
        layers = [nn.Linear(2, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, xy):
        return self.net(xy)


def d_d(f, v, idx):
    g = torch.autograd.grad(f, v, grad_outputs=torch.ones_like(f),
                             create_graph=True, retain_graph=True)[0]
    return g[:, idx:idx + 1]


N_int = 2000
xy_int = torch.rand(N_int, 2, device=device).requires_grad_(True)

N_bnd = 200
edges = []
for val, axis in [(0.0, 0), (1.0, 0), (0.0, 1), (1.0, 1)]:
    t = torch.rand(N_bnd // 4)
    pts = torch.zeros(N_bnd // 4, 2); pts[:, axis] = val; pts[:, 1 - axis] = t
    edges.append(pts)
xy_bnd = torch.cat(edges, dim=0).to(device)

xy_fk = torch.tensor(xy_fk_np, dtype=torch.float32, device=device)
u_fk_t = torch.tensor(u_fk_labels, dtype=torch.float32, device=device).view(-1, 1)
f_int = torch.tensor(f_source(xy_int[:, 0:1].detach().cpu().numpy(),
                               xy_int[:, 1:2].detach().cpu().numpy()),
                      dtype=torch.float32, device=device)


def compute_loss(model, use_fk, lam_bnd=10.0, lam_fk=1.0):
    u = model(xy_int)
    u_x = d_d(u, xy_int, 0); u_y = d_d(u, xy_int, 1)
    u_xx = d_d(u_x, xy_int, 0); u_yy = d_d(u_y, xy_int, 1)
    r_pde = torch.mean((-(u_xx + u_yy) - f_int)**2)

    r_bnd = torch.mean(model(xy_bnd)**2)

    loss = r_pde + lam_bnd * r_bnd
    if use_fk:
        r_fk = torch.mean((model(xy_fk) - u_fk_t)**2)
        loss = loss + lam_fk * r_fk
    return loss

## 4. Entrenamiento: PINN estandar vs. FK-PINN

In [ ]:
def train(use_fk, epochs=4000, lr=1e-3):
    model = PoissonPINN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = []
    for epoch in range(epochs):
        opt.zero_grad()
        loss = compute_loss(model, use_fk)
        loss.backward()
        opt.step()
        hist.append(loss.item())
        if epoch % 1000 == 0:
            print(f'[{"FK-PINN" if use_fk else "PINN"}] epoch {epoch:5d} | loss={loss.item():.4e}')
    return model, hist


model_pinn, hist_pinn = train(use_fk=False)
model_fk, hist_fk = train(use_fk=True)

## 5. Resultados: precision y velocidad de convergencia (cf. Fig. de resultados de Poisson, Apendice E)

In [ ]:
n_side = 60
xs = np.linspace(0, 1, n_side); ys = np.linspace(0, 1, n_side)
Xg, Yg = np.meshgrid(xs, ys)
xy_test = torch.tensor(np.stack([Xg.ravel(), Yg.ravel()], axis=1), dtype=torch.float32, device=device)
u_ex = exact_u(Xg, Yg)

with torch.no_grad():
    u_pinn = model_pinn(xy_test).cpu().numpy().reshape(Xg.shape)
    u_fk = model_fk(xy_test).cpu().numpy().reshape(Xg.shape)

err_pinn = 100 * np.linalg.norm(u_pinn - u_ex) / np.linalg.norm(u_ex)
err_fk = 100 * np.linalg.norm(u_fk - u_ex) / np.linalg.norm(u_ex)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].semilogy(hist_pinn, label='PINN estandar')
axes[0].semilogy(hist_fk, label='FK-PINN')
axes[0].set_xlabel('Epoca'); axes[0].set_ylabel('Loss (escala log)')
axes[0].set_title('Convergencia del entrenamiento'); axes[0].legend()

axes[1].bar(['PINN estandar', 'FK-PINN'], [err_pinn, err_fk], color=['tab:orange', 'tab:blue'])
axes[1].set_ylabel('Error relativo L2 (%)')
axes[1].set_title('Precision final')
plt.tight_layout()
plt.show()

print(f'Error relativo L2 -- PINN estandar: {err_pinn:.2f}%  |  FK-PINN: {err_fk:.2f}%')